# EduCrew Studio: free AI episode maker
**Series:** EduCrew: From a Little Dream to a Bigger World (10 episodes, 3D animated, 9:16, 60 s)

**One-time setup:** in Settings, set Accelerator to **GPU T4 x2**, turn **Internet** on, and set **Persistence** to **Files only** (if shown).
In Add-ons → Secrets, add `HF_TOKEN` (your Hugging Face *Read* token) and tick it.

**Every session:**
1. In the **Setup** cell, set `EPISODE`, then run it (Shift + Enter).
2. Run **Steps 1 → 5** in order, one at a time. Each step skips anything already made, so re-running is always safe.

Episodes must be made **in order** (01 → 10): each one starts from the previous episode's last picture.
The character pictures (`EduCrew_Characters.json`) only need Setup + Step 1, once.

In [ ]:
# ▶ WHICH VIDEO TO MAKE (change only this line)
EPISODE = "episodes/EduCrew_Biography_Episode_01.json"

import glob, os, shutil
PROJECT = "/kaggle/working/educrew-studio"
if not os.path.exists(PROJECT):
    !git clone -q https://github.com/educatorscircleec-dot/educrew-studio {PROJECT}
# always use the latest fixes from GitHub (your pictures, clips and voices are kept)
!cd {PROJECT} && git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
os.chdir(PROJECT)
NAME = os.path.splitext(os.path.basename(EPISODE))[0]
WORK = f"work/{NAME}"
for p in glob.glob("/kaggle/input/**/educrew_logo.png", recursive=True):
    shutil.copy(p, "logo/educrew_logo.png")
for z in glob.glob("/kaggle/input/**/work.zip", recursive=True):
    shutil.unpack_archive(z, PROJECT); print("restored", z)

# install everything once per session (skipped if already installed)
try:
    import compel, av, open_clip, torchdiffeq, psutil
except ImportError:
    !pip install -q -U diffusers transformers accelerate ftfy imageio-ffmpeg sentencepiece "compel>=2.5" psutil av colorlog einops librosa omegaconf open_clip_torch timm torchdiffeq soundfile
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

from IPython.display import display, Image, Video, Audio
def show(kind):
    """show("keyframes"), show("clips"), show("voice"), show("sfx")"""
    ext = {"keyframes": "png", "clips": "mp4", "voice": "wav", "sfx": "wav"}[kind]
    for p in sorted(glob.glob(f"{WORK}/{kind}/*.{ext}")):
        print(p)
        display(Image(p, width=220) if ext == "png" else Video(p, embed=True, width=220) if ext == "mp4" else Audio(p))
print("Ready to make:", NAME)

## Step 1: pictures (one per shot, about 10 min)
To redo one picture, see **Redo one thing** at the bottom.

In [ ]:
!cd tools && python make_keyframes.py ../$EPISODE
show("keyframes")

## Step 2: video clips (slowest: about 40–50 min for 10 shots on both GPUs)
Both GPUs work at the same time. If the session stops, just run this cell again; finished clips are kept.

In [ ]:
!cd tools && python make_clips.py ../$EPISODE
show("clips")

## Step 3: voices (narrator, family, exact words, about 10 min)

In [ ]:
!python tools/setup_tts_env.py
!cd tools && /kaggle/working/.tts_env/bin/python make_voices.py ../$EPISODE
show("voice")

## Step 4: sound effects (natural sounds only, no music, about 10 min)

In [ ]:
if not os.path.exists("/kaggle/working/MMAudio"):
    !git clone -q https://github.com/hkchengrex/MMAudio /kaggle/working/MMAudio
!cd tools && python make_sfx.py ../$EPISODE --mmaudio /kaggle/working/MMAudio
show("sfx")

## Step 5: the finished 60-second Reel (about 2 min)
Download it from the **Output** panel: `educrew-studio/output/`.

In [ ]:
!python assemble.py $EPISODE
Video(glob.glob(f"output/*{NAME}.mp4")[0], embed=True, width=300)

## Redo one thing (only if needed)
Remove the `#` in front of the line you need, change the shot number, and run this cell.
A different `--seed` number gives a different version.

In [ ]:
# !cd tools && python make_keyframes.py ../$EPISODE --shots 3 --force --seed 77
# !cd tools && python make_clips.py ../$EPISODE --shots 3 --force --seed 77
# !cd tools && /kaggle/working/.tts_env/bin/python make_voices.py ../$EPISODE --shots 3 --force --seed 21
# !cd tools && python make_sfx.py ../$EPISODE --shots 3 --force --seed 5

## Save progress
Only needed if Persistence is **not** on. Then download `work.zip` from **Output**, and next time attach it with **Add Input**.

In [ ]:
shutil.make_archive("/kaggle/working/work", "zip", PROJECT, "work")
print("saved /kaggle/working/work.zip")